In [1]:

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report


In [2]:

# ============================================================
# 0. Reproducibility
# ============================================================

SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)



In [4]:


## df = pd.read_csv("cyber_logs.csv")


df = pd.read_csv("cyber_dataset_C.csv")



In [5]:

df


,prompt,tool_ground_truth,tool_predicted,pred_svm,source,row_id
0,The vendor gave us a jump-box address and temp...,SSHConnect,SSHConnect,SSHConnect,pilot,C0000
1,Can you establish a terminal session to the Ub...,SSHConnect,SSHConnect,SSHConnect,pilot,C0001
2,We moved the inventory appliance to a new subn...,NmapScan,NmapScan,NmapScan,pilot,C0002
3,Does 172.16.9.25 still advertise that old SSH ...,NmapScan,NmapScan,NmapScan,pilot,C0003
4,The change request says the database node shou...,PortScan,PortScan,PortScan,pilot,C0004
...,...,...,...,...,...,...
1495,Read the local journal around the fan-control ...,ReadSyslog,ReadAuthLog,ReadSyslog,batch495,C1510
1496,The VPN interface renamed itself on this lapto...,ReadSyslog,GetSystemInfo,ReadSyslog,batch495,C1511
1497,After we rolled out a new eBPF-based observabi...,ReadSyslog,ReadSyslog,ReadSyslog,batch495,C1512
1498,Get the operating-system journal entries from ...,ReadSyslog,ReadSyslog,ReadSyslog,batch495,C1513


In [6]:

print("Rows:", len(df))
print(df.columns.tolist())


Rows: 1500
['prompt', 'tool_ground_truth', 'tool_predicted', 'pred_svm', 'source', 'row_id']


In [7]:

print("\nGround-truth tools:")
print(df["tool_ground_truth"].value_counts())



Ground-truth tools:
tool_ground_truth
SSHConnect            137
NmapScan              137
BlockIP               137
GetSystemInfo         137
PortScan              136
CheckVulnerability    136
ReadAuthLog           136
CheckFailedLogins     136
ListeningPorts        136
ListProcesses         136
ReadSyslog            136
Name: count, dtype: int64


In [ ]:

# Create disagreement label:
# 0 = LLM prediction agrees with annotation
# 1 = LLM prediction disagrees with annotation


In [8]:

df["disagreement"] = (
    df["pred_svm"] != df["tool_ground_truth"]
).astype(int)


In [9]:

df


,prompt,tool_ground_truth,tool_predicted,pred_svm,source,row_id,disagreement
0,The vendor gave us a jump-box address and temp...,SSHConnect,SSHConnect,SSHConnect,pilot,C0000,0
1,Can you establish a terminal session to the Ub...,SSHConnect,SSHConnect,SSHConnect,pilot,C0001,0
2,We moved the inventory appliance to a new subn...,NmapScan,NmapScan,NmapScan,pilot,C0002,0
3,Does 172.16.9.25 still advertise that old SSH ...,NmapScan,NmapScan,NmapScan,pilot,C0003,0
4,The change request says the database node shou...,PortScan,PortScan,PortScan,pilot,C0004,0
...,...,...,...,...,...,...,...
1495,Read the local journal around the fan-control ...,ReadSyslog,ReadAuthLog,ReadSyslog,batch495,C1510,0
1496,The VPN interface renamed itself on this lapto...,ReadSyslog,GetSystemInfo,ReadSyslog,batch495,C1511,0
1497,After we rolled out a new eBPF-based observabi...,ReadSyslog,ReadSyslog,ReadSyslog,batch495,C1512,0
1498,Get the operating-system journal entries from ...,ReadSyslog,ReadSyslog,ReadSyslog,batch495,C1513,0


In [10]:

print("\nAgreement / disagreement:")
print(df["disagreement"].value_counts())



Agreement / disagreement:
disagreement
0    1378
1     122
Name: count, dtype: int64


In [11]:

# ============================================================
# 2. TF-IDF
# ============================================================



tfidf = TfidfVectorizer(
    ngram_range=(1, 2)
)

X = tfidf.fit_transform(df["prompt"]).toarray()

feature_names = tfidf.get_feature_names_out()

print("\nTF-IDF shape:", X.shape)
print("Number of TF-IDF features:", len(feature_names))




TF-IDF shape: (1500, 25151)
Number of TF-IDF features: 25151


In [12]:

X


array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [13]:

np.set_printoptions(threshold=np.inf)

print(feature_names)


['00' '00 and' '00 to' '01' '01 before' '01 it' '02' '02 10' '02 get'
 '02 my' '024' '024 tcp' '03' '03 00' '03 internal' '03 show' '04' '04 12'
 '04 as' '04 but' '04 only' '04 we' '06' '06 30' '07' '07 30' '07 from'
 '09' '09 10' '09 17' '09 30' '10' '10 01' '10 10' '10 11' '10 118'
 '10 12' '10 14' '10 15' '10 17' '10 18' '10 20' '10 22' '10 23' '10 24'
 '10 30' '10 31' '10 32' '10 33' '10 34' '10 38' '10 40' '10 42' '10 44'
 '10 50' '10 55' '10 60' '10 61' '10 62' '10 70' '10 73' '10 77' '10 80'
 '10 91' '10 and' '10 build' '10 retrieve' '100' '100 104' '100 108'
 '100 11' '100 118' '100 126' '100 14' '100 142' '100 151' '100 162'
 '100 167' '100 174' '100 177' '100 186' '100 187' '100 197' '100 203'
 '100 213' '100 216' '100 225' '100 234' '100 239' '100 24' '100 249'
 '100 25' '100 31' '100 32' '100 38' '100 42' '100 44' '100 47' '100 49'
 '100 59' '100 61' '100 64' '100 67' '100 71' '100 73' '100 82' '100 83'
 '100 86' '100 94' '100 96' '100 percent' '1000' '1000 tcp' '10001'
 '1

In [14]:

# ============================================================
# 3. Encode the 11 ground-truth tools
# ============================================================

label_encoder = LabelEncoder()

y_tool = label_encoder.fit_transform(
    df["tool_ground_truth"]
)

tool_names = label_encoder.classes_

print("\nTools:")
for i, tool in enumerate(tool_names):
    print(i, tool)



Tools:
0 BlockIP
1 CheckFailedLogins
2 CheckVulnerability
3 GetSystemInfo
4 ListProcesses
5 ListeningPorts
6 NmapScan
7 PortScan
8 ReadAuthLog
9 ReadSyslog
10 SSHConnect


In [15]:


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_tool,
    test_size=0.20,
    random_state=SEED,
    stratify=y_tool
)



In [16]:


X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)

y_train_t = torch.tensor(y_train, dtype=torch.long)
y_test_t  = torch.tensor(y_test,  dtype=torch.long)



In [17]:

X_train_t.shape


torch.Size([1200, 25151])

In [18]:

y_train_t.shape


torch.Size([1200])

In [19]:


class ToolNet(nn.Module):

    def __init__(self, input_dim, num_classes):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(input_dim, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, num_classes)
        )

    def forward(self, x):

        return self.net(x)


In [20]:

model_tool = ToolNet(
    input_dim=X.shape[1],
    num_classes=len(tool_names)
)


In [21]:

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model_tool.parameters(),
    lr=0.001
)



In [22]:

EPOCHS = 200

model_tool.train()

for epoch in range(EPOCHS):

    optimizer.zero_grad()

    logits = model_tool(X_train_t)

    loss   = criterion(logits, y_train_t)

    loss.backward()

    optimizer.step()

    if (epoch + 1) % 20 == 0:

        pred = logits.argmax(dim=1)

        acc = (
            pred == y_train_t
        ).float().mean().item()

        print(
            f"Epoch {epoch+1:3d} | "
            f"Loss = {loss.item():.4f} | "
            f"Train Accuracy = {acc:.4f}"
        )



Epoch  20 | Loss = 2.1771 | Train Accuracy = 1.0000
Epoch  40 | Loss = 1.3914 | Train Accuracy = 1.0000
Epoch  60 | Loss = 0.3372 | Train Accuracy = 1.0000
Epoch  80 | Loss = 0.0515 | Train Accuracy = 1.0000
Epoch 100 | Loss = 0.0176 | Train Accuracy = 1.0000
Epoch 120 | Loss = 0.0102 | Train Accuracy = 1.0000
Epoch 140 | Loss = 0.0071 | Train Accuracy = 1.0000
Epoch 160 | Loss = 0.0054 | Train Accuracy = 1.0000
Epoch 180 | Loss = 0.0042 | Train Accuracy = 1.0000
Epoch 200 | Loss = 0.0034 | Train Accuracy = 1.0000


In [23]:



model_tool.eval()

with torch.no_grad():

    logits = model_tool(X_test_t)

    y_pred = logits.argmax(dim=1).cpu().numpy()


print("\n==========================================")
print("TOOL CLASSIFICATION")
print("==========================================")

print(
    "Test Accuracy:",
    accuracy_score(y_test, y_pred)
)

print(
    classification_report(
        y_test,
        y_pred,
        target_names=tool_names
    )
)





TOOL CLASSIFICATION
Test Accuracy: 0.85
                    precision    recall  f1-score   support

           BlockIP       0.93      0.93      0.93        28
 CheckFailedLogins       0.96      0.85      0.90        27
CheckVulnerability       1.00      0.93      0.96        27
     GetSystemInfo       0.76      0.96      0.85        27
     ListProcesses       0.85      0.81      0.83        27
    ListeningPorts       0.82      0.52      0.64        27
          NmapScan       0.96      0.86      0.91        28
          PortScan       0.62      0.74      0.68        27
       ReadAuthLog       0.86      0.89      0.87        27
        ReadSyslog       0.85      0.85      0.85        27
        SSHConnect       0.82      1.00      0.90        28

          accuracy                           0.85       300
         macro avg       0.86      0.85      0.85       300
      weighted avg       0.86      0.85      0.85       300



In [24]:


# Maximum observed value for every TF-IDF feature.
#
# NIO will NOT be allowed to produce negative TF-IDF values
# or values larger than those actually observed in the corpus.

feature_max = torch.tensor(
    X.max(axis=0),
    dtype=torch.float32
)



In [25]:

feature_max


tensor([0.4237, 0.1414, 0.2238,  ..., 0.2735, 0.1368, 0.1368])

In [26]:


# ============================================================
# 9. Freeze trained tool model
# ============================================================

for param in model_tool.parameters():

    param.requires_grad = False

model_tool.eval()


ToolNet(
  (net): Sequential(
    (0): Linear(in_features=25151, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=11, bias=True)
  )
)

In [27]:

def nio_tool(
    model,
    target_class,
    feature_max,
    steps=500,
    lr=0.05,
    norm_weight=0.001
):

    # Start near zero inside valid TF-IDF range

    x_opt = torch.rand( 1,  len(feature_max)  )

    x_opt = x_opt * feature_max.unsqueeze(0)

    x_opt.requires_grad_(True)

    optimizer_x = optim.Adam(  [x_opt], lr=lr  )

    target = torch.tensor(  [target_class], dtype=torch.long  )


    for step in range(steps):

        optimizer_x.zero_grad()

        logits = model(x_opt)

        loss_target = criterion( logits, target  )

        # Encourage a relatively small TF-IDF vector
        loss_norm = x_opt.pow(2).mean()

        loss = (  loss_target + norm_weight * loss_norm  )

        loss.backward()
        optimizer_x.step()


        # --------------------------------------------
        # Keep optimized input inside observed
        # TF-IDF feature ranges
        # --------------------------------------------

        with torch.no_grad():

            x_opt.clamp_(min=0.0)

            x_opt.copy_( torch.minimum( x_opt, feature_max.unsqueeze(0)  ) )


    return x_opt.detach()




In [28]:


# ============================================================
# 11. Extract top TF-IDF features
# ============================================================

def print_top_features( x_opt, feature_names, top_n=10 ):

    values = (  x_opt.cpu().numpy()[0]  )

    indices = np.argsort(values)[::-1]

    print()

    count = 0

    for idx in indices:

        if values[idx] <= 0:
            continue

        print(
            f"{feature_names[idx]:30s} "
            f"{values[idx]:.4f}"
        )

        count += 1

        if count >= top_n:
            break




In [29]:


# ============================================================
# NIO EXPERIMENT 1
#
#     What TF-IDF features characterize each tool?
# ============================================================

print("\n\n")
print("==========================================")
print("NIO EXPERIMENT 1")
print("TOOL CHARACTERIZATION")
print("==========================================")

nio_tool_results = {}

for tool_index, tool_name in enumerate(tool_names):

    print("\n------------------------------------------")
    print("TARGET TOOL:", tool_name)
    print("------------------------------------------")

    x_star = nio_tool(
        model=model_tool,
        target_class=tool_index,
        feature_max=feature_max,
        steps=500,
        lr=0.05
    )

    nio_tool_results[tool_name] = x_star

    print_top_features(
        x_star,
        feature_names,
        top_n=10
    )







NIO EXPERIMENT 1
TOOL CHARACTERIZATION

------------------------------------------
TARGET TOOL: BlockIP
------------------------------------------

with memory                    0.3651
running as                     0.3612
18 at                          0.3605
processes sorted               0.3601
list processes                 0.3578
bulletins for                  0.3504
suspicious access              0.3489
affect rabbitmq                0.3461
grafana                        0.3447
block 203                      0.3401

------------------------------------------
TARGET TOOL: CheckFailedLogins
------------------------------------------

need snapshot                  0.4341
00                             0.4236
of running                     0.4168
tasks belong                   0.4151
what eating                    0.4095
eating ram                     0.4095
ram on                         0.4095
pytest processes               0.4066
pytest                         0.4066
leftover


## Target vs. Pred

* On agreement vs. disagreement label


In [30]:


# ============================================================
# Create disagreement dataset
# ============================================================

y_error = df["disagreement"].values

X_train_e, X_test_e, y_train_e, y_test_e = train_test_split(
    X,
    y_error,
    test_size=0.20,
    random_state=SEED,
    stratify=y_error
)

X_train_e = torch.tensor( X_train_e, dtype=torch.float32 )

X_test_e = torch.tensor(   X_test_e, dtype=torch.float32 )

y_train_e = torch.tensor( y_train_e, dtype=torch.long    )

y_test_e = torch.tensor(   y_test_e, dtype=torch.long    )



In [31]:

# ============================================================
# 14. Disagreement neural network
# ============================================================

class ErrorNet(nn.Module):

    def __init__(self, input_dim):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(input_dim, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 2)
        )

    def forward(self, x):

        return self.net(x)


In [32]:


model_error = ErrorNet(    input_dim=X.shape[1]    )

optimizer_error = optim.Adam(  model_error.parameters(),  lr=0.001    )


In [33]:


model_error.train()

for epoch in range(EPOCHS):

    optimizer_error.zero_grad()

    logits = model_error(  X_train_e   ) 

    loss = criterion(  logits,  y_train_e   )

    loss.backward()

    optimizer_error.step()

    if (epoch + 1) % 20 == 0:

        pred = logits.argmax(  dim=1  )

        acc = (   pred == y_train_e   ).float().mean().item()

        print(
            f"Epoch {epoch+1:3d} | "
            f"Loss = {loss.item():.4f} | "
            f"Train Accuracy = {acc:.4f}"
        )



Epoch  20 | Loss = 0.4212 | Train Accuracy = 0.9292
Epoch  40 | Loss = 0.0704 | Train Accuracy = 0.9975
Epoch  60 | Loss = 0.0124 | Train Accuracy = 1.0000
Epoch  80 | Loss = 0.0018 | Train Accuracy = 1.0000
Epoch 100 | Loss = 0.0009 | Train Accuracy = 1.0000
Epoch 120 | Loss = 0.0006 | Train Accuracy = 1.0000
Epoch 140 | Loss = 0.0004 | Train Accuracy = 1.0000
Epoch 160 | Loss = 0.0003 | Train Accuracy = 1.0000
Epoch 180 | Loss = 0.0002 | Train Accuracy = 1.0000
Epoch 200 | Loss = 0.0002 | Train Accuracy = 1.0000


In [34]:


model_error.eval()

with torch.no_grad():

    logits = model_error(X_test_e)

    pred_error = (  logits.argmax(dim=1).cpu().numpy()  )


print("\n==========================================")
print("DISAGREEMENT CLASSIFICATION")
print("==========================================")

print( "Accuracy:", accuracy_score( y_test_e.numpy(), pred_error ) )

print(
    classification_report(
        y_test_e.numpy(),
        pred_error,
        target_names=[
            "Agreement",
            "Disagreement"
        ]
    )
)




DISAGREEMENT CLASSIFICATION
Accuracy: 0.93
              precision    recall  f1-score   support

   Agreement       0.93      1.00      0.96       276
Disagreement       1.00      0.12      0.22        24

    accuracy                           0.93       300
   macro avg       0.96      0.56      0.59       300
weighted avg       0.93      0.93      0.90       300



In [35]:


for param in model_error.parameters():

    param.requires_grad = False

model_error.eval()



ErrorNet(
  (net): Sequential(
    (0): Linear(in_features=25151, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=2, bias=True)
  )
)

In [36]:


# ============================================================
# NIO EXPERIMENT 2
#
# What TF-IDF features characterize disagreement between
# predicted tool and annotated ground-truth tool?
# ============================================================

print("\n\n")
print("==========================================")
print("NIO EXPERIMENT 2")
print("PREDICTION DISAGREEMENT")
print("==========================================")

# Class 1 = disagreement

x_error_star = nio_tool(
    model=model_error,
    target_class=1,
    feature_max=feature_max,
    steps=500,
    lr=0.05
)

print("\nNIO features associated with disagreement:")

print_top_features(
    x_error_star,
    feature_names,
    top_n=20
)






NIO EXPERIMENT 2
PREDICTION DISAGREEMENT

NIO features associated with disagreement:

ram on                         0.4077
binding 5000                   0.4059
what binding                   0.4059
binding                        0.3843
socket for                     0.3706
5000                           0.3690
for 5432                       0.3676
know what                      0.3571
out this                       0.3558
which pid                      0.3557
maxing out                     0.3556
pid is                         0.3555
affect rabbitmq                0.3457
reserving tcp                  0.3453
24800 on                       0.3452
24800                          0.3448
busiest tasks                  0.3403
by cpu                         0.3403
tasks by                       0.3387
consuming the                  0.3371


In [38]:


# ============================================================
# NIO EXPERIMENT 2.b
#
# What TF-IDF features characterize disagreement between
# predicted tool and annotated ground-truth tool?
# ============================================================

print("\n\n")
print("==========================================")
print("NIO EXPERIMENT 2")
print("PREDICTION AGREEMENT")
print("==========================================")

# Class 0 = agreement

x_error_star = nio_tool(
    model=model_error,
    target_class=0,
    feature_max=feature_max,
    steps=500,
    lr=0.05
)

print("\nNIO features associated with agreement:")

print_top_features(
    x_error_star,
    feature_names,
    top_n=20
)





NIO EXPERIMENT 2
PREDICTION AGREEMENT

NIO features associated with agreement:

of running                     0.2097
libarchive                     0.2060
binds                          0.2051
command lines                  0.2048
for openldap                   0.2045
the processor                  0.2037
for libarchive                 0.2036
advisory trail                 0.2029
pytest                         0.2020
openldap                       0.2019
openldap 59                    0.2019
bulletins for                  0.2017
check for                      0.2016
processes piling               0.2011
need snapshot                  0.2011
leftover                       0.2010
pytest processes               0.2010
leftover pytest                0.2006
any defunct                    0.2004
hogging                        0.2003


In [39]:

errors = df[df["disagreement"] == 1]

print(
    errors.groupby(
        ["tool_ground_truth", "tool_predicted"]
    ).size().sort_values(ascending=False)
)


tool_ground_truth   tool_predicted    
PortScan            PortScan              15
ReadAuthLog         ReadAuthLog           14
NmapScan            NmapScan              13
CheckFailedLogins   CheckFailedLogins     11
GetSystemInfo       GetSystemInfo         11
ReadSyslog          ReadSyslog             9
ListeningPorts      ListeningPorts         9
CheckVulnerability  CheckVulnerability     8
ListProcesses       ListProcesses          6
BlockIP             BlockIP                4
ListeningPorts      PortScan               4
                    NmapScan               4
PortScan            NmapScan               3
NmapScan            CheckVulnerability     2
ReadSyslog          ReadAuthLog            2
ListProcesses       GetSystemInfo          2
SSHConnect          SSHConnect             2
ListeningPorts      CheckVulnerability     1
ListProcesses       PortScan               1
ReadSyslog          GetSystemInfo          1
dtype: int64
